In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv
/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv


# Introduction

This notebook contains code for a micro-Small Language Model (< 2 Million parameters) consisting of a Language Model with 1-2 transformer blocks and multi-headed, causal self-attention. This model is pretrained on the TinyStories dataset for the next-token prediction task. The pretrained model is then repeatedly input generated text to generate a short story.

# Steps:
1. **Data Exploration**
2. **Data Preprocessing**
     1. Tokenization
     2. Saving data to memory
     3. Input-target pairing
3. **Building Model Architecture**
    1. Projection to Token Embedding Space
    2. Projection to Positional Embedding Space (for later; unclear on this)
    3. Attention
    4. Feed-forward Network/ Multi-layer Perceptrons
    5. Layer Normalization
    6. Transformer
    7. Small Language Model (contains embedding projection, transformer blocks, LM Head, etc.)
4. **Pre-training**
    1. Defining cross-entropy loss function
    2. Updating parameters using backpropagation
    3. Pre-training using training data
    4. Validate model on validation dataset
    5. Visualize and analyze training and validation losses
    6. Hyperparameter tuning and repeat training and validation
5. **Short Story Generation**
6. **Explore and Evaluate Generated Output**


# Step 1: Data Exploration

In [2]:
if not os.path.exists('./data'):
    os.makedirs('./data')
    
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'train.csv', output_dir = './data')
kagglehub.dataset_download('thedevastator/tinystories-narrative-classification',
                           path = 'validation.csv', output_dir = './data')

'/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv'

In [3]:
train = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/train.csv')
val = pd.read_csv('/kaggle/input/datasets/thedevastator/tinystories-narrative-classification/validation.csv')

In [4]:
pd.options.display.max_rows = 500

print("Summary of training dataset:")
print(train.describe)

Summary of training dataset:
<bound method NDFrame.describe of                                                       text
0        One day, a little girl named Lily found a need...
1        Once upon a time, there was a little car named...
2        One day, a little fish named Fin was swimming ...
3        Once upon a time, in a land full of trees, the...
4        Once upon a time, there was a little girl name...
...                                                    ...
2119714  Once upon a time, in a small town, there lived...
2119715  Once upon a time, there was a little boy named...
2119716  Once upon a time, there was a big tree. Under ...
2119717  Once upon a time, there was a little girl name...
2119718  Once upon a time, there was an adorable little...

[2119719 rows x 1 columns]>


In [5]:
print("Summary of validation dataset:")
print(val.describe)

Summary of validation dataset:
<bound method NDFrame.describe of                                                     text
0      Spot. Spot saw the shiny car and said, "Wow, K...
1      Once upon a time, in a big forest, there lived...
2      Once upon a time, in a small yard, there was a...
3      Once upon a time, there was a thoughtful girl ...
4      Once upon a time, there was a kind farmer. He ...
...                                                  ...
21985  John was out in the backyard playing. He saw a...
21986  Once upon a time, there was a green vehicle. E...
21987  Once upon a time, there was a package. It was ...
21988  Once upon a time, there was a thin rhinoceros....
21989  Once there was a little girl named Sarah. She ...

[21990 rows x 1 columns]>


In [6]:
print('First story in training set:')
print(train.iloc[0]['text'])

First story in training set:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


## Select sample of 10 stories for initial run 

In [7]:
print('2 more stories from training set and validation set each:')
train_sample = train.sample(10, random_state = 1)
val_sample = val.sample(2, random_state = 1)

print(train_sample.iloc[0]['text'])
print('-------------X-------------')
print(train_sample.iloc[1]['text'])
print('-------------X-------------')
print(val_sample.iloc[0]['text'])
print('-------------X-------------')
print(val_sample.iloc[1]['text'])

2 more stories from training set and validation set each:
Once upon a time, there was a big elephant named Ellie. Ellie loved to walk in the forest and eat leaves. One day, Ellie met a little bird named Billy. Billy said, "Hi, Ellie! You are so big and graceful. Can I walk with you?"

Ellie was happy to have a friend to walk with. They walked and walked until they saw a big snake. The snake was not happy and said, "Get out of my way! I am going to poke you with my sharp teeth!"

Ellie and Billy were scared and did not know what to do. Ellie said, "Please don't poke us. We just want to walk." The snake thought for a moment and said, "Okay, you can walk. But be careful, there are many dangers in the forest."

Ellie and Billy were happy to continue their walk and thanked the snake for being kind. They walked until they reached a river. Ellie said, "Let's poke our trunks in the water and have a drink." They drank the water and continued their walk, happy to have each other's company.
-----

## Tokenize Data and Create Chunks of Input-Target Pairs

Firstly, we must break the natural language text into units called **tokens** and assign a numerical Id to each called a **token ID**. The list of containing all the units in the language, be they words, sub-words, or byte-pairs, indexed by their corresponding token IDs is called the **vocabulary**. For this task, we will use a pre-existing GPT2 tokenizer to perform **Byte-Pair Encoding (BPE)**.

For the next token prediction task, a **sliding window** is defined, such that given the tokens in the window, i.e. the **context**, the next token in the sequence is considered to be the expected output. The max number of tokens that can be in the window is the **context size**. Tokenized data is grouped into these input (the tokens in the window) and target (the next expected token) pairs. These groups are called **chunks**.

In [8]:
!pip install git+https://github.com/huggingface/transformers

  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-9406n6eb
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-9406n6eb
  Resolved https://github.com/huggingface/transformers to commit f02ee386724df0096e9e0bb011a840a3463b769b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 16.4 MB/s eta 0:00:0000:01
  Created wheel for transformers: filename=transformers-5.24.0.dev0-py3-none-any.whl size=14718182 sha256=5c09eb311ad3cae4f742060da4e816c00f619613334ad3e95bf464422c89415a
  Stored in directory: /tmp/pip-ephem-wheel-cache-yq2fs6dd/wheels/2b/de/48/1c5b158806820c3979e1dc7a341b68dac1231f00b1a2c9442f
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.29.0
    Uninstalling huggingface_hub-1.2

In [9]:
import torch.nn.functional as F
import torch
from transformers import GPT2Tokenizer, BatchEncoding

In [10]:
class Dataset:
    def __init__(self, rawData, contextSize, split):
        self.split = split
        tokenizer = GPT2Tokenizer.from_pretrained('openai-community/gpt2', add_eos_token = True)
        tokenData = tokenizer(rawData)
        self.vocabSize = tokenizer.vocab_size
        tokenIds = []
        for document in tokenData['input_ids']:
            tokenIds.extend(document)
        # Chunking tokens based on input and target
        self.inputTokens = torch.zeros((len(tokenIds) - contextSize,contextSize), dtype = torch.int64)
        self.targetTokens = torch.zeros((len(tokenIds) - contextSize,contextSize), dtype = torch.int64)
        for i in range(len(tokenIds) - contextSize):
            try:
                self.inputTokens[i,:] = torch.tensor(tokenIds[i:i+contextSize])
                self.targetTokens[i,:] = torch.tensor(tokenIds[i+1:i+contextSize+1])
            except:
                print("i = ", i)
                print("input vector = ", torch.tensor(tokenIds[i:i+contextSize]))
                print("target vector = ", torch.tensor(tokenIds[i+1:i+contextSize+1]))
                break

    def __len__(self):
        return self.inputTokens.size(dim=0)

    def __getitem__(self, idx):
        return self.inputTokens[idx,:], self.targetTokens[idx,:]

In [11]:
trainData = Dataset(train_sample['text'].tolist(), contextSize = 4, split = 'train')
valData = Dataset(val_sample['text'].tolist(), contextSize = 4, split = 'val')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

### Batch Data to Prepare it for the Embedding Layer of the Model

In [12]:
# an iterable class to batch data in dataset and iterated through the batches
class DataLoader:
    def __init__(self, dataset, batchSize, contextSize):
        self.dataLen = dataset.__len__()
        assert self.dataLen % batchSize == 0
        # Batching data from Dataset object
        self.batchSize = batchSize
        self.contextSize = contextSize
        self.numOfBatches = self.dataLen / batchSize
        self.currentBatch = 0
        self.dataset = dataset

    def __iter__(self):
        self.currentBatch = 0
        return self

    def __next__(self):
        if self.currentBatch >= self.numOfBatches:
            raise StopIteration
        inputData = torch.zeros((self.batchSize, self.contextSize), dtype = torch.int64)
        targetData = torch.zeros((self.batchSize, self.contextSize), dtype = torch.int64)
        # get data for current batch and collate it
        index = self.currentBatch * self.batchSize
        for i in range(self.batchSize):
            try:
                inputData[i,:], targetData[i,:] = self.dataset[index + i] # syntactical sugar for __getitem__()
            except:
                print("Error at index = ", index, " currentBatch = ", self.currentBatch, " i = ", i)
                break
        self.currentBatch = self.currentBatch + 1
        return inputData, targetData

In [13]:
dataloader = DataLoader(trainData, 25, 4)
for batchId, batch in enumerate(dataloader):
    if batchId % 30 == 0:
        print("Batch No. = ", batchId + 1)
        print("inputData slice = ", batch[0][0:3, :], "targetData slice = ", batch[1][0:3,:])
        print("----------------------------------------")

Batch No. =  1
inputData slice =  tensor([[7454, 2402,  257,  640],
        [2402,  257,  640,   11],
        [ 257,  640,   11,  612]]) targetData slice =  tensor([[2402,  257,  640,   11],
        [ 257,  640,   11,  612],
        [ 640,   11,  612,  373]])
----------------------------------------
Batch No. =  31
inputData slice =  tensor([[ 588,  484, 4888,  262],
        [ 484, 4888,  262, 1097],
        [4888,  262, 1097,   13]]) targetData slice =  tensor([[ 484, 4888,  262, 1097],
        [4888,  262, 1097,   13],
        [ 262, 1097,   13,  198]])
----------------------------------------
Batch No. =  61
inputData slice =  tensor([[1327,  673, 3111,   11],
        [ 673, 3111,   11,  340],
        [3111,   11,  340, 1239]]) targetData slice =  tensor([[ 673, 3111,   11,  340],
        [3111,   11,  340, 1239],
        [  11,  340, 1239, 3947]])
----------------------------------------
Batch No. =  91
inputData slice =  tensor([[ 356, 3124,  674, 5916],
        [3124,  674, 5916,

## Project to Embedding Space

A set of numbers, such as these **tokens**, carry no information whatsoever to a computer. To imbue these tokens with some **semantic meaning**, we project them to a higher order **embedding space**. Thus, each token becomes a vector of *n* dimensions, where *n* is the dimension of the embedding space.

These weights, first initialized randomly, will be learned by the Language Model during training.

In [14]:
class Dropout:
    def __init__(self, drop):
        self.drop = drop

    def forward(self, inputEmbd):
        # shape(inputEmbd) = (batchSize, contextSize, embdDim)
        mask = torch.rand(size=inputEmbd.shape) < self.drop
        return inputEmbd.masked_fill_(mask, 0)

In [15]:
class LayerNorm:
    def __init__(self, embdDim):
        self.embdDim = embdDim
        self.eps = 1e-5

    def forward(self, inputEmbd):
        # shape(inputEmbd) = (batchSize, contextSize, embdDim)
        mean = torch.sum(inputEmbd, dim = 2) / self.embdDim # shape(mean) = (batchSize, contextSize)
        dev = inputEmbd.permute(2, 0, 1) - mean
        dev = dev.permute(1, 2, 0) # shape(dev) = shape(batchSize, contextSize, embdDim)
        devSq = dev ** 2
        var = torch.sum(devSq, dim = 2) / self.embdDim # shape(var) = shape(batchSize, contextSize)
        normEmbd = dev.permute(2, 0, 1) / ((var + self.eps) ** 0.5)
        return normEmbd.permute(1, 2, 0) # shape(normEmbd) = (batchSize, contextSize, embdDim)

In [16]:
from math import sqrt

class Attention:
    def __init__(self, attnHeads, dimK, embdDim, drop):
        self.attnHeads = attnHeads
        self.dimK = dimK
        self.embdDim = embdDim
        self.weightsQ = torch.rand(embdDim, embdDim) 
        self.weightsK = torch.rand(embdDim, embdDim)
        self.weightsV = torch.rand(embdDim, embdDim)
        self.weightsO = torch.rand(embdDim, embdDim)
        self.dropout = Dropout(drop)

    def forward(self, inputEmbd, batchSize, contextSize):
        # reshape query and key from (batchSize, contextSize, embdDim)
        query = torch.reshape(inputEmbd @ self.weightsQ, 
                             (batchSize, contextSize, self.attnHeads, self.dimK))
        key = torch.reshape(inputEmbd @ self.weightsK,
                           (batchSize, contextSize, self.attnHeads, self.dimK))
        value = torch.reshape(inputEmbd @ self.weightsV,
                             (batchSize, contextSize, self.attnHeads, self.dimK))

        # rearranging axes of query and key to prepare them for scalar multiplication
        # new shape should be (batchSize, attnHeads, contextSize, dimK)
        query = query.permute(0, 2, 1, 3)
        key = key.permute(0, 2, 1, 3)
        causalMask = torch.tril(torch.ones((contextSize, contextSize)))
        dotProduct = query @ key.transpose(2, 3) # shape of dotProduct is (contextSize, contextSize)
        maskedDotProduct = dotProduct.masked_fill_(causalMask == 0, float('-inf'))
        scaledDotProduct = torch.softmax(maskedDotProduct/ sqrt(self.dimK), dim = 2)
        attnWeights = self.dropout.forward(scaledDotProduct) @ value.permute(0, 2, 1, 3)
        attnWeights = torch.reshape(attnWeights.permute(0, 2, 1, 3),
                                    (batchSize, contextSize, self.embdDim)) # use contiguous instead
        return attnWeights @ self.weightsO

In [17]:
import torch.nn.functional as F

class MLP:
    def __init__(self, embdDim, projDim, drop):
        self.projDim = projDim
        self.weightsUp = torch.rand(embdDim, projDim)
        self.weightsDown = torch.rand(projDim, embdDim)
        self.dropout = Dropout(drop)

    def forward(self, inputEmbd):
        # shape(inputEmbd) = (batchSize, contextSize, embdDim)
        inputProj = inputEmbd @ self.weightsUp # shape(inputProj) = (batchSize, contextSize, projDim)
        output = inputProj @ self.weightsDown # shape(output) = (batchSize, contextSize, embdDim)
        return self.dropout.forward(F.gelu(output))

In [18]:
class Transformer:
    def __init__(self, embdDim, attnHeads, dimK, projDim, drop):
        self.layerNorm = LayerNorm(embdDim)
        self.attention = Attention(attnHeads, dimK, embdDim, drop)
        self.mlp = MLP(embdDim, projDim, drop)

    def forward(self, inputEmbd, batchSize, contextSize):
        output = self.layerNorm.forward(self.attention.forward(inputEmbd, batchSize, contextSize)) 
        + inputEmbd # adding residual
        output = self.layerNorm.forward(self.mlp.forward(output)) + output # adding residual
        return output

In [19]:
class TinyGPT:
    def __init__(self, config):
        self.contextSize = config['contextSize']
        self.batchSize = config['batchSize']
        self.vocabSize = config['vocabSize']
        self.embdDim = config['embdDim']
        self.attnHeads = config['attnHeads']
        self.dimK = config['dimK']
        self.drop = config['drop']
        self.projDim = config['projDim']
        self.embedMatrix = torch.rand(config['vocabSize'], config['embdDim'])

        self.transformers = []
        for i in range(config['transformerBlocks']):
            self.transformers.append(Transformer(self.embdDim, self.attnHeads, self.dimK, self.projDim, self.drop))

        self.finalLayerNorm = LayerNorm(config['embdDim'])
        self.weightsOut = torch.rand(config['embdDim'], config['vocabSize'])
    
    def embedding_layer(self, inputTokens): # perform lookup in Embedding Matrix
        return self.embedMatrix[inputTokens] # still have to add positional embedding

    def forward(self, embd):
        for transformer in self.transformers:
            embd = transformer.forward(embd, self.batchSize, self.contextSize)
        embd = self.finalLayerNorm.forward(embd)
        # Language Modelling Head
        logits = embd @ self.weightsOut
        return logits

In [20]:
config = {
            'contextSize': 4,
            'vocabSize': trainData.vocabSize,
            'batchSize': 25,
            'embdDim': 128,
            'attnHeads': 2,
            'dimK': 64,
            'drop': 0.2,
            'projDim': 256,
            'transformerBlocks': 2
        }
model = TinyGPT(config)

In [57]:
def compute_cost(logits, targets, batchSize):
    targets = torch.flatten(targets, start_dim = 0, end_dim = 1)
    probs = torch.flatten(torch.softmax(logits, dim = 2), start_dim = 0, end_dim = 1)
    # print("shape of targets = ", targets.shape, ", shape of probs = ", probs.shape)
    targetProbs = torch.gather(probs, 1, targets.unsqueeze_(dim = 1))
    return torch.mean(torch.log(targetProbs), dim = 0) * -1 # cross-entropy loss
     

In [58]:
for batchId, batch in enumerate(dataloader):
    tokenEmbd = model.embedding_layer(batch[0])
    logits = model.forward(tokenEmbd)
    if batchId % 50 == 0:
        # print("batch ", batchId, ": transformer output = ", output)
        print("batch ", batchId, ": shape of output = ", logits.shape,
              ", shape of target = ", batch[1].shape)
        loss = compute_cost(logits, batch[1], config['batchSize'])
        print("loss = ", loss)
# print("The embedding of the first token is ")
# print(tokenEmbd[0, 0, :])

batch  0 : shape of output =  torch.Size([25, 4, 50257]) , shape of target =  torch.Size([25, 4])
loss =  tensor([14.7668])
batch  50 : shape of output =  torch.Size([25, 4, 50257]) , shape of target =  torch.Size([25, 4])
loss =  tensor([13.3406])
batch  100 : shape of output =  torch.Size([25, 4, 50257]) , shape of target =  torch.Size([25, 4])
loss =  tensor([14.1151])
